In [9]:
import pandas as pd
from pathlib import Path    
from statsmodels.stats.multitest import multipletests
from scipy import stats
import numpy as np

In [10]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
metrics = ["AUROC"]
less_bias_strengths = ["0.1"]
method_name_replacer = {"mrs-forest": "MRS",  
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        }
data_set_replacer = {"diabetes": "Diabetes",
                    "folktables_employment": "Employment", 
                     "folktables_income": "Income",
                     "bank_marketing": "Bank Marketing",
                     "hr_analytics": "HR Analytic",
                     "german_credit": "German Credit", 
                     "breast_cancer": "Breast Cancer", 
                     "loan_prediction": "Loan",
                    }

In [11]:
method_pairs = []
for other_method in ("fw-mrs-temperature", "fw-mrs-temperature-svm"):
    method_pairs.append(("mrs-forest", other_method))
method_pairs

[('mrs-forest', 'fw-mrs-temperature'),
 ('mrs-forest', 'fw-mrs-temperature-svm')]

In [12]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            for bias_strength in less_bias_strengths:
                json_directory = result_path / dataset / bias_type /  bias_strength/ method / "classification_results"
                auroc_file = pd.read_json(str(json_directory / "rf_auroc_list.json"))
                dict_list.append(
                    {
                        "Method": method,
                        "Data Set": dataset,
                        "AUROC": auroc_file.values,
                        "Bias Type": bias_type,
                        "Bias Strength": bias_strength
                    }
                                  )
result_df = pd.DataFrame(data=dict_list)

In [13]:
result_df.explode("AUROC")

,Method,Data Set,AUROC,Bias Type,Bias Strength
0,mrs-forest,diabetes,[0.783965484431123],less_positive_class,0.1
0,mrs-forest,diabetes,[0.815353005542472],less_positive_class,0.1
0,mrs-forest,diabetes,[0.785377863412545],less_positive_class,0.1
0,mrs-forest,diabetes,[0.7856692594971331],less_positive_class,0.1
0,mrs-forest,diabetes,[0.765619915873419],less_positive_class,0.1
...,...,...,...,...,...
23,fw-mrs-temperature-svm,loan_prediction,[0.32373737373737305],less_positive_class,0.1
23,fw-mrs-temperature-svm,loan_prediction,[0.525],less_positive_class,0.1
23,fw-mrs-temperature-svm,loan_prediction,[0.6606060606060601],less_positive_class,0.1
23,fw-mrs-temperature-svm,loan_prediction,[0.6592897581060211],less_positive_class,0.1


In [14]:
def corrected_t_test(first_values, second_values, n_folds=5.0):
    differences = first_values - second_values
    mean_differences = np.mean(differences)
    std_differences = np.std(differences)
    train_size = n_folds - 1.0
    test_size = 1.0 
    correction_factor = (1.0 / len(differences)) + (test_size / train_size)
    return mean_differences / (np.sqrt(correction_factor) * std_differences)

In [15]:
p_values = []

for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
            for bias_strength in less_bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        first_metrics = result_df.loc[(result_df["Method"]==first_method_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        second_metrics = result_df.loc[(result_df["Method"]==second_metric_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        df = len(first_metrics) - 1
                        t_statistic = corrected_t_test(np.squeeze(first_metrics), np.squeeze(second_metrics))
                        p_values.append((1.0 - stats.t.cdf(abs(t_statistic), df)) * 2.0)
corrected_p_values = multipletests(p_values, method="fdr_bh")

In [16]:
i = 0
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
            for bias_strength in less_bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        print(f"p value for {metric}, {dataset}, {bias_type}, {bias_strength}, {first_method_name},\
{second_metric_name} is: {corrected_p_values[0][i]}, {corrected_p_values[1][i]}")
                        i += 1

p value for AUROC, diabetes, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.6951441714726018
p value for AUROC, diabetes, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.6951441714726018
p value for AUROC, folktables_employment, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.6879647925926526
p value for AUROC, folktables_employment, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.33610145592941265
p value for AUROC, folktables_income, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.8030682406136148
p value for AUROC, folktables_income, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.33610145592941265
p value for AUROC, bank_marketing, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.8188141529146058
p value for AUROC, bank_marketing, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.6879647925926526
p valu